In [4]:
%run notebook_init.py

from rag_pipeline.retriever.retrieve import query_chunks
from rag_pipeline.llm.generator import generate_answer_with_gate

In [ ]:
# LAYER 3 RETRIEVAL PIPELINE

# Test 1: Location query (should hit ITEM 2 / Properties)
chunks = query_chunks("Where is Tesla headquartered?", top_k=5)
assert len(chunks) > 0, "No chunks returned"
assert all("text" in c and "metadata" in c for c in chunks), "Chunks missing fields"

# Test 2: Risk query (should hit ITEM 1A)
chunks = query_chunks("What are Tesla's main risk factors?", top_k=5)
assert any("1A" in c["metadata"].get("section", "").upper() or "RISK" in c["metadata"].get("section", "").upper() for c in chunks), "Risk query didn't retrieve risk section"

# Test 3: Nonsense query (should still return chunks, not crash)
chunks = query_chunks("asdfghjkl", top_k=5)
# Should return something (even if irrelevant), not throw

/Users/aditya/Documents/projects/hallucination-resistant-finance-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEBUG: Single-doc mode: True
DEBUG: Detected intents: ['HQ_LOCATION'], confidence: 0.50
DEBUG: Entity terms: ['tesla']
DEBUG: Intent synonyms: ['locate', 'city', 'located', 'based', 'head office']...
🔍 Loaded embedding model: sentence-transformers/paraphrase-MiniLM-L3-v2
DEBUG: Semantic coverage: has_coverage=True, intent_matches=1, evidence_hits=0
DEBUG: Use keyword fallback: True
DEBUG: Top keyword matches (score, intent, evidence, entity, has_evidence, length):
  [1] ITEM 1. BUSINESS: score=42.80 (intent=4, evidence=0, entity=1, has_evidence=False, len=2699)
  [2] ITEM 1. BUSINESS: score=35.85 (intent=3, evidence=1, entity=1, has_evidence=True, len=2747)
  [3] ITEM 1A. RISK FACTORS: score=26.04 (intent=2, evidence=1, entity=1, has_evidence=True, len=2937)
DEBUG: Keyword fallback found 208 matches
DEBUG: Final result_chunks count=5
DEBUG: Total context chars=12274
  Final chunk 1: ITEM 1. BUSINESS (2699 chars)
  Final chunk 2: ITEM 1. BUSINESS (2747 chars)
  Final chunk 3: ITEM 1A. R

In [6]:
# LAYER 4 GENERATION PIPELINE

chunks = query_chunks("Where is Tesla headquartered?", top_k=5)
answer, abstained = generate_answer_with_gate("Where is Tesla headquartered?", chunks)

print(f"Answer: {answer}")
print(f"Abstained: {abstained}")

chunks = query_chunks("asdfghjkl", top_k=5)
answer, abstained = generate_answer_with_gate("asdfghjkl", chunks)

print(f"Answer: {answer}")
print(f"Abstained: {abstained}")

DEBUG: Single-doc mode: True
DEBUG: Detected intents: ['HQ_LOCATION'], confidence: 0.50
DEBUG: Entity terms: ['tesla']
DEBUG: Intent synonyms: ['locate', 'city', 'located', 'based', 'head office']...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]


DEBUG: Semantic coverage: has_coverage=True, intent_matches=1, evidence_hits=0
DEBUG: Use keyword fallback: True
DEBUG: Top keyword matches (score, intent, evidence, entity, has_evidence, length):
  [1] ITEM 1. BUSINESS: score=42.80 (intent=4, evidence=0, entity=1, has_evidence=False, len=2699)
  [2] ITEM 1. BUSINESS: score=35.85 (intent=3, evidence=1, entity=1, has_evidence=True, len=2747)
  [3] ITEM 1A. RISK FACTORS: score=26.04 (intent=2, evidence=1, entity=1, has_evidence=True, len=2937)
DEBUG: Keyword fallback found 208 matches
DEBUG: Final result_chunks count=5
DEBUG: Total context chars=12274
  Final chunk 1: ITEM 1. BUSINESS (2699 chars)
  Final chunk 2: ITEM 1. BUSINESS (2747 chars)
  Final chunk 3: ITEM 1A. RISK FACTORS (2937 chars)
  Final chunk 4: ITEM 1A. RISK FACTORS (2746 chars)
  Final chunk 5: ITEM 2. PROPERTIES (1145 chars)
✅ Retrieved 5 relevant chunks (requested 5)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:rag_pipeline:Query: Where is Tesla headquartered?
INFO:rag_pipeline:  [1] tesla-2024-10K::ITEM 1. BUSINESS::chunk-23
INFO:rag_pipeline:  [2] tesla-2024-10K::ITEM 1. BUSINESS::chunk-36
INFO:rag_pipeline:  [3] tesla-2024-10K::ITEM 1A. RISK FACTORS::chunk-46
INFO:rag_pipeline:  [4] tesla-2024-10K::ITEM 1A. RISK FACTORS::chunk-60
INFO:rag_pipeline:  [5] tesla-2024-10K::ITEM 2. PROPERTIES::chunk-71
INFO:rag_pipeline:Answer length: 44 chars


Answer: Tesla is headquartered in Austin, Texas [5].
Abstained: False
DEBUG: Single-doc mode: True
DEBUG: Detected intents: [], confidence: 0.00
DEBUG: Entity terms: []
DEBUG: Intent synonyms: []...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.91it/s]


DEBUG: Semantic coverage: has_coverage=False, intent_matches=0, evidence_hits=0
DEBUG: Use keyword fallback: False
DEBUG: Final result_chunks count=5
DEBUG: Total context chars=10699
  Final chunk 1: ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA (2495 chars)
  Final chunk 2: ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA (2881 chars)
  Final chunk 3: Item 16. (2025 chars)
  Final chunk 4: ITEM 10. DIRECTORS, EXECUTIVE OFFICERS AND CORPORATE GOVERNANCE (512 chars)
  Final chunk 5: ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA (2786 chars)
✅ Retrieved 5 relevant chunks (requested 5)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:rag_pipeline:Query: asdfghjkl
INFO:rag_pipeline:  [1] tesla-2024-10K::ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA::chunk-137 (distance=27.4996)
INFO:rag_pipeline:  [2] tesla-2024-10K::ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA::chunk-113 (distance=27.9941)
INFO:rag_pipeline:  [3] tesla-2024-10K::Item 16.::chunk-18 (distance=29.0674)
INFO:rag_pipeline:  [4] tesla-2024-10K::ITEM 10. DIRECTORS, EXECUTIVE OFFICERS AND CORPORATE GOVERNANCE::chunk-181 (distance=29.4879)
INFO:rag_pipeline:  [5] tesla-2024-10K::ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA::chunk-122 (distance=29.5896)
INFO:rag_pipeline:Answer length: 56 chars


Answer: The provided document does not contain that information.
Abstained: False
